In [1]:
import joblib
import numpy as np
import pandas as pd

# 1) Model ML + urutan fitur
model = joblib.load("model_medagent.pkl")
fitur = joblib.load("fitur_gejala.pkl")

# 2) Info penyakit (untuk severity) dari sheet "Info Penyakit"
info_df = pd.read_excel("dataset.xlsx", sheet_name="Info Penyakit")
info_penyakit = info_df.set_index("Nama Penyakit").to_dict("index")

# 3) Aturan tindakan per tingkat severity
aturan_tindakan = {
    "HIGH":     {"warna": "🔴 Merah",  "tindakan": "SEGERA ke IGD / UGD rumah sakit terdekat.", "urgensi": "Darurat — butuh penanganan medis segera."},
    "MODERATE": {"warna": "🟡 Kuning", "tindakan": "Periksa ke dokter dalam 24 jam.",           "urgensi": "Perlu perhatian — jangan ditunda lama."},
    "LOW":      {"warna": "🟢 Hijau",  "tindakan": "Istirahat di rumah & pantau gejala. Ke dokter bila memburuk.", "urgensi": "Ringan — umumnya bisa dirawat di rumah."},
}

print("Semua komponen berhasil dimuat ✅")

Semua komponen berhasil dimuat ✅


In [2]:
def buat_input(gejala_pasien):
    baris = {f: (1 if f in gejala_pasien else 0) for f in fitur}
    return pd.DataFrame([baris])[fitur]   # [fitur] menjaga urutan kolom

def ambil_info_penyakit(nama_penyakit):
    data = info_penyakit.get(nama_penyakit)
    if data is None:
        return {"error": f"Info untuk '{nama_penyakit}' tidak ditemukan."}
    severity = data["Risk Severity"]
    tindakan = aturan_tindakan.get(severity, {})
    return {
        "kategori": data["Kategori"],
        "deskripsi": data["Deskripsi"],
        "dokter": data["Dokter Spesialis"],
        "severity": severity,
        "warna": tindakan.get("warna"),
        "urgensi": tindakan.get("urgensi"),
        "tindakan": tindakan.get("tindakan"),
    }

In [3]:
def triase(gejala_pasien, threshold=0.40):
    # 1) Model menebak penyakit
    X = buat_input(gejala_pasien)
    proba = model.predict_proba(X)[0]
    classes = model.classes_
    top3_idx = np.argsort(proba)[::-1][:3]

    penyakit_utama = classes[top3_idx[0]]
    confidence = float(proba[top3_idx[0]])
    top3 = [(classes[i], round(float(proba[i]), 3)) for i in top3_idx]

    # 2) Severity memberi konteks
    konteks = ambil_info_penyakit(penyakit_utama)

    # 3) Gabung jadi satu laporan
    return {
        "penyakit_utama": penyakit_utama,
        "confidence": round(confidence, 3),
        "keyakinan_rendah": confidence < threshold,
        "top3": top3,
        **konteks,
    }

In [4]:
def cetak_laporan(gejala_pasien):
    h = triase(gejala_pasien)
    print("=" * 50)
    print("LAPORAN TRIASE MEDAGENT")
    print("=" * 50)
    print("Gejala    :", ", ".join(gejala_pasien))
    print(f"Penyakit  : {h['penyakit_utama']} ({h['confidence']*100:.1f}%)")
    if h["keyakinan_rendah"]:
        print("⚠️  Keyakinan rendah — gejala mungkin terlalu umum.")
    print("Severity  :", h["warna"], h["severity"])
    print("Urgensi   :", h["urgensi"])
    print("Tindakan  :", h["tindakan"])
    print("Dokter    :", h["dokter"])
    print("Top-3     :", h["top3"])

# tes
cetak_laporan(["fever", "chills", "headache"])
cetak_laporan(["chest_pain", "shortness_of_breath", "orthopnea", "swollen_feet"])

LAPORAN TRIASE MEDAGENT
Gejala    : fever, chills, headache
Penyakit  : Leptospirosis (32.0%)
⚠️  Keyakinan rendah — gejala mungkin terlalu umum.
Severity  : 🔴 Merah HIGH
Urgensi   : Darurat — butuh penanganan medis segera.
Tindakan  : SEGERA ke IGD / UGD rumah sakit terdekat.
Dokter    : Spesialis Penyakit Dalam / Sp. PD
Top-3     : [('Leptospirosis', 0.32), ('Influenza', 0.225), ('Pneumonia', 0.075)]
LAPORAN TRIASE MEDAGENT
Gejala    : chest_pain, shortness_of_breath, orthopnea, swollen_feet
Penyakit  : Heart Failure (50.0%)
Severity  : 🔴 Merah HIGH
Urgensi   : Darurat — butuh penanganan medis segera.
Tindakan  : SEGERA ke IGD / UGD rumah sakit terdekat.
Dokter    : Spesialis Jantung / Sp. JP
Top-3     : [('Heart Failure', 0.5), ('Asthma', 0.405), ('GERD', 0.03)]
